# ETF 公司行动数据资格

按已冻结 [plan.json](plan.json) 离线核对发行人现金分配、拆股份额单位与参考日历。来源来自原始发行人 XLS、明确标记的浏览器表格转存、原公告和历史存档；失败响应同样封存。原生 Qlib Calendar 与父 Recorder 是证据入口。本卡无模型训练、策略回测或晋升，金额源字符串与未知时间均保留。

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "AGENTS.md").exists() and (p / "SPEC.md").exists())
sys.path.insert(0, str(ROOT))
import qlib
from qlib.data import D
from qlib.workflow import R
from src.workflow.etf_corporate_actions import (assert_run_available, read_frozen_plan, run_etf_corporate_actions_qualification)
CARD = ROOT / "research/etf_corporate_actions_20261009"
SOURCE_ROOT = ROOT
EVIDENCE_ROOT = ROOT
EXPECTED_PLAN_SHA256 = "2127d65862abb8c0b17178ba35a6694ead94af27d11099e4f0ddcc32806492eb"
assert_run_available(CARD)
plan = read_frozen_plan(CARD / "plan.json", expected_plan_sha256=EXPECTED_PLAN_SHA256)
reference = plan["calendar_reference"]
qlib.init(provider_uri=str(EVIDENCE_ROOT / reference["provider_uri"]), region=reference["region"], kernels=1)
calendar = D.calendar(freq=reference["freq"], future=False)
print({"native_calendar_rows": len(calendar), "as_of": plan["market_as_of"], "scope": plan["task_kind"]})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[90557:MainThread](2026-10-09 15:22:11,381) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[90557:MainThread](2026-10-09 15:22:11,383) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[90557:MainThread](2026-10-09 15:22:11,383) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxx_extended_published_affine_20260925_v1')}


{'native_calendar_rows': 5021, 'as_of': '2026-09-25', 'scope': 'event_source_and_share_unit_qualification_only'}


## 来源、父证据与事件资格

入口先拒绝完成卡片或活跃 Recorder，再验证冻结计划。逐哈希独立复制所有冻结来源；分别显式使用新、旧父 manifest 和对象哈希读取已完成 Recorder。SOXX 原 XLS 重解析与库存 CSV 用 Decimal 精确比较，原 strings 不被替换。历史发行人存档的原始份额金额及日期须和现有 SOXL 行一致。全部事件、差异、原始来源、父绑定及参考日历仅保存到 Recorder。

In [2]:
summary = run_etf_corporate_actions_qualification(
    CARD / "plan.json", output_card=CARD, source_root=SOURCE_ROOT, evidence_root=EVIDENCE_ROOT,
    expected_plan_sha256=EXPECTED_PLAN_SHA256,
)
summary

[90557:MainThread](2026-10-09 15:22:11,393) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[90557:MainThread](2026-10-09 15:22:11,393) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[90557:MainThread](2026-10-09 15:22:11,394) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxx_extended_published_affine_20260925_v1')}


[90557:MainThread](2026-10-09 15:22:11,547) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name etf_corporate_actions_20261009.


[90557:MainThread](2026-10-09 15:22:11,549) INFO - qlib.workflow - [exp.py:258] - Experiment 649036106770763940 starts running ...


[90557:MainThread](2026-10-09 15:22:11,629) INFO - qlib.workflow - [recorder.py:345] - Recorder 582a0c7d39964e8b860ebe33941c5dd2 starts running under Experiment 649036106770763940 ...


[90557:MainThread](2026-10-09 15:22:12,576) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'experiment': 'etf_corporate_actions_20261009',
 'record_type': 'ETF_event_source_qualification',
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/soxx_extended_published_affine_20260925_v1',
 'market': 'all',
 'region': 'us',
 'as_of': '2026-09-25',
 'data_range': ['2006-10-10', '2026-09-25'],
 'native_calendar_rows': 5021,
 'cash_event_rows': 125,
 'split_event_rows': 3,
 'source_file_count': 50,
 'source_archive': 'data/raw/external/etf_corporate_actions_20261009_v1',
 'qualification_pass': True,
 'decision': 'usable_under_declared_source_contract',
 'plan_sha256': '2127d65862abb8c0b17178ba35a6694ead94af27d11099e4f0ddcc32806492eb',
 'source_sha256': {'src/workflow/etf_corporate_actions.py': '6b8f8832511ac0c0a54ea0e0e6ea5717e0b0435898cb9bbfd3d30d1828e3c9f9',
  'src/data/etf_corporate_actions.py': '5544f3f87969065f03880b3c95c231fbac831a28bb659f6043e5c6ac6f689e7e',
  'src/core/recorder_sources.py': '3a6a15935247d3f4552fd99cc6b0b0fc33f2edadf5f9195527079312c710b3f1'},
 'recorder

## 资格结果与边界

[qualification.json](qualification.json) 为事件数与原始来源核验摘要；[source_differences_summary.json](source_differences_summary.json) 保留差异分类，逐行差额在 Recorder。[summary.json](summary.json) 仅在归档成功、Recorder FINISHED 及哈希读回后发布。

In [3]:
from IPython.display import Markdown, display
display(Markdown((CARD / "conclusion.md").read_text()))

# ETF 公司行动数据资格

已离线核验 125 条现金事件、3 条拆股及 5021 日原生 Qlib 参考日历；原始来源 50 文件逐字节封存。

SOXX 较早事件保留为参考日历外，零金额事件保留；窗口差异属于数据覆盖，不是收益表现。SOXX 2017 漏项、SOXL 2014 份额冲突与 2019 精度差保留在 Recorder。发行人存档直接证明 2014-12-23 每原始份额 0.00532，但抓取晚于除息日，不能证明事前获知。期末未支付分红仍是应收，不能当可支出现金。

本卡未拟合、未回测、未改变旧价格，无晋升或稳定超额结论。下一步须另卡登记真实份额、应收及支付现金账本，保留到账时点与税费未知项。

Recorder：582a0c7d39964e8b860ebe33941c5dd2。


## 中文结论

本 notebook 执行时核对原始来源字节、SOXX/SOXL 全量现金事件、有效拆股与原生参考日历；上方结论输出中的行数、来源差异和 Recorder 身份为本次运行证据。早于参考日历的行与零金额行均保留，期末未支付项目仍是应收而非可支出现金。数据覆盖窗口差异不解释为收益差异。

不晋升。本卡未训练模型、未运行策略、未改变旧价格；当前历史获取与除息日后的存档不能构成事前 PIT 或真实账户到账认证。下一步另卡登记真实份额、应收与 pay-date 现金账本；详细边界见执行输出及 [conclusion.md](conclusion.md)。